# Setpoint Feedforward (Anticipation de la consigne)

Simulation of the GEL-7063 exercise model `ex_anticipation_consigne.slx`
using the python-control library.

The control system is a 2-DOF structure with model-based setpoint
feedforward:

$$
U(s) = G_{ff}(s) R(s) + G_c(s) \left( G_r(s) R(s) - Y(s) \right)
$$

$$
Y(s) = G_p(s) U(s) + D(s)
$$

where:
 - $G_p(s)$ : plant (process)
 - $G_m(s)$ : model of the plant, used to design $G_{ff}(s)$ and $G_r(s)$
 - $G_c(s)$ : PI feedback controller
 - $G_{ff}(s)$ : setpoint feedforward
 - $G_r(s)$ : reference model (desired setpoint response)
 - $D(s)$ : output disturbance

In [ ]:
import control as con
import matplotlib.pyplot as plt
import numpy as np

## Plant

$$
G_p(s) = \frac{3(-2s + 1)}{(10s + 1)(5s + 1)} e^{-4s}
$$

python-control transfer functions are rational, so the time delay is
kept separate from the rational part $G_{p,0}(s)$. For simulation, the
models are converted to discrete time, where the delay is represented
exactly as a shift of $\theta / T_s$ samples (see
[Discretization](#Discretization)).

In [ ]:
s = con.tf('s')

# Plant parameters
K_p = 3.0      # gain
T_1 = 10.0     # dominant time constant (s)
T_2 = 5.0      # secondary time constant (s)
T_z = 2.0      # RHP zero time constant (s)
theta = 4.0    # time delay (s)

# Rational part of plant
G_p0 = K_p * (-T_z * s + 1) / ((T_1 * s + 1) * (T_2 * s + 1))
print(G_p0)

## Discretization

All blocks are converted to discrete time with a zero-order hold (ZOH)
at sample time $T_s$. The time delay is then an exact shift of
$n_d = \theta / T_s$ samples:

$$
G_p(z) = G_{p,0}(z) \, z^{-n_d}
$$

The other blocks are discretized in the same way where they are defined
below.

The ZOH conversion is exact for blocks whose input is constant between
samples (e.g. the setpoint path when $r(t)$ steps at a sample time). In
the feedback loop the results are an approximation of the continuous
system, with errors that decrease with $T_s$ (about 0.003 in $y(t)$ for
$T_s = 0.1$ s).

In [ ]:
# Sample time
Ts = 0.1

# Time delay as a shift of n_d samples
n_d = round(theta / Ts)
assert np.isclose(n_d * Ts, theta), "delay must be a multiple of Ts"
z_delay = con.ss(con.tf([1], [1] + [0] * n_d, Ts))

# Discrete-time plant (ZOH)
G_p_d = z_delay * con.ss(con.c2d(G_p0, Ts, 'zoh'))
print(f"Delay: {n_d} samples")

## Closed-Loop System

The function below constructs the closed-loop system from its component
transfer functions. Each block is given named input and output signals,
and the blocks are connected with `con.interconnect`:

```none
         ┌─────────┐ u_ff
     ┌──►│ G_ff(s) │──────────────────────────────┐
     │   └─────────┘                              │                     d
     │                                            │                     │
     │                                            │                     │
     │   ┌─────────┐ r_m      e ┌────────┐ u_fb   ▼+ u  ┌────────┐ y_p  ▼+
r ───┴──►│ F_r(s)  │─────►(Σ)──►│ G_c(s) │──────►(Σ)───►│ G_p(s) │────►(Σ)──┬──► y
         └─────────┘     + ▲−   └────────┘      +       └────────┘    +     │
                           │                                                │
                           └────────────────────────────────────────────────┘
```

Only the feedback controller $G_c$ and the plant $G_p$ are required. The
other blocks are optional:

| Argument | Block | Default |
|---|---|---|
| `F_r` | Reference filter, $r \rightarrow r_m$ | 1 (no filter) |
| `G_ff` | Setpoint feedforward, $r \rightarrow u_{ff}$ | `None` (no feedforward) |

The disturbance $d$ is added directly to the plant output.

For the setpoint feedforward design above, the reference filter is the
reference model, $F_r = G_r$.

In [ ]:
def make_closed_loop_system(G_c, G_p, F_r=1, G_ff=None, name='closed_loop'):
    """Construct a closed-loop system with optional setpoint feedforward.

    All systems must have the same sample time as G_p (or all be
    continuous-time). Scalars are treated as static gains.

    Args:
        G_c: Feedback controller, e -> u_fb.
        G_p: Plant, u -> y_p.
        F_r: Reference filter, r -> r_m (default 1, no filter).
        G_ff: Setpoint feedforward, r -> u_ff (default None, no
            feedforward).
        name: Name of the closed-loop system.

    Returns:
        Closed-loop system with inputs (r, d) and outputs
        (r_m, e, u, y), where d is an output disturbance.
    """
    dt = G_p.dt

    def block(sys, inputs, outputs, name):
        if np.isscalar(sys):
            sys = con.tf(sys, 1, dt)
        return con.ss(sys, inputs=inputs, outputs=outputs, name=name)

    blocks = [
        block(F_r, 'r', 'r_m', 'F_r'),
        block(G_c, 'e', 'u_fb', 'G_c'),
        block(G_p, 'u', 'y_p', 'G_p'),
        con.summing_junction(inputs=['r_m', '-y'], output='e', dt=dt, name='sum_e'),
        con.summing_junction(inputs=['y_p', 'd'], output='y', dt=dt, name='sum_y'),
    ]
    if G_ff is None:
        u_inputs = ['u_fb']
    else:
        blocks.append(block(G_ff, 'r', 'u_ff', 'G_ff'))
        u_inputs = ['u_ff', 'u_fb']
    blocks.append(
        con.summing_junction(inputs=u_inputs, output='u', dt=dt, name='sum_u')
    )

    return con.interconnect(
        blocks, inputs=['r', 'd'], outputs=['r_m', 'e', 'u', 'y'], name=name
    )

## Simulation

Simulation parameters from the Simulink model:

| Parameter | Value |
|---|---|
| Simulation time | 0 – 200 s |
| Setpoint step $r(t)$ | 0 → 1 at $t = 1$ s |
| Disturbance step $d(t)$ | 0 → 1 at $t = 100$ s |
| Initial conditions | zero |

Simulink uses a variable-step solver (RelTol = 1e-3) with exact transport
delays. Here the discrete-time closed-loop systems are simulated at the
sample times $t = k T_s$.

In [ ]:
# Simulation parameters
t_stop = 200.0
t_r, r_step = 1.0, 1.0      # setpoint step time and size
t_d, d_step = 100.0, 1.0    # disturbance step time and size

t = Ts * np.arange(round(t_stop / Ts) + 1)
r = np.where(t >= t_r - Ts / 2, r_step, 0.0)
d = np.where(t >= t_d - Ts / 2, d_step, 0.0)


def simulate(T):
    """Simulate closed-loop system T with the inputs (r, d).

    Returns:
        Dictionary of output signals (r_m, e, u, y).
    """
    sim = con.forced_response(T, T=t, U=[r, d])
    return {name: sim.outputs[T.find_output(name)] for name in T.output_labels}

In [ ]:
def plot_sim_results(t, r, sim, title=None):
    """Plot the setpoint, output, error and control signals."""
    fig, axes = plt.subplots(3, 1, sharex=True, figsize=(7, 1 + 3*1.5))

    ax = axes[0]
    ax.plot(t, sim['y'], label='$y(t)$')
    ax.plot(t, r, label='$r(t)$')
    ax.plot(t, sim['r_m'], '--', label='$r_m(t)$')
    ax.set_ylabel('$r(t)$, $y(t)$')
    ax.set_title(title)
    ax.legend()

    ax = axes[1]
    ax.plot(t, sim['e'])
    ax.set_ylabel('$e(t)$')

    ax = axes[2]
    ax.plot(t, sim['u'])
    ax.set_ylabel('$u(t)$')
    ax.set_xlabel('Temps (s)')

    for ax in axes:
        ax.grid()
    plt.tight_layout()
    plt.show()

## PI Feedback Controller

$$
G_c(s) = K_c \frac{T_i s + 1}{T_i s}
$$

In [ ]:
# Controller parameters
K_c = 0.21
T_i = 10.0

G_c = K_c * (T_i * s + 1) / (T_i * s)
G_c_d = con.c2d(G_c, Ts, 'zoh')
print(G_c)

### Simulation 1: Feedback Control Only

The closed-loop system with the PI controller only: no reference filter
($F_r = 1$) and no setpoint feedforward.

In [ ]:
T_fb = make_closed_loop_system(G_c_d, G_p_d, name='feedback_only')
sim_fb = simulate(T_fb)
plot_sim_results(t, r, sim_fb, title='Feedback control only')

## Reference Filter

The reference filter $F(s)$ specifies the desired setpoint tracking
dynamics (user choice):

$$
F(s) = \frac{1}{(5s + 1)^2}
$$

In [ ]:
# Reference filter parameters
T_F = 5.0      # filter time constant (s)

F = 1 / (T_F * s + 1) ** 2
F_d = con.c2d(F, Ts, 'zoh')
print(F)

### Simulation 2: Reference Filter

The setpoint is filtered by $F(s)$ before the error is computed
($F_r = F$), with no setpoint feedforward.

In [ ]:
T_filt = make_closed_loop_system(G_c_d, G_p_d, F_r=F_d, name='ref_filter')
sim_filt = simulate(T_filt)
plot_sim_results(t, r, sim_filt, title='Reference filter')

## Setpoint Feedforward and Reference Model

The setpoint feedforward is designed from the reference filter $F(s)$
and a model of the plant.

### Plant Model Decomposition

The model of the plant is assumed to be perfect, $G_m(s) = G_p(s)$. It
is decomposed into invertible and non-invertible parts:

$$
G_m(s) = \tilde{G}_m(s) \bar{G}_m(s)
= \frac{3}{(10s + 1)(5s + 1)} \cdot (-2s + 1) e^{-4s}
$$

The non-invertible part $\bar{G}_m(s)$ contains the RHP zero and the time
delay. As before, the delay is kept separate from the rational part
$\bar{G}_{m,0}(s) = -2s + 1$.

In [ ]:
# Model of the plant (perfect model, rational part)
G_m0 = G_p0

# Invertible part
G_m_inv = K_p / ((T_1 * s + 1) * (T_2 * s + 1))

# Non-invertible part (rational part)
G_m_ninv0 = -T_z * s + 1

# Check that the two parts recombine to the model rational part
diff = con.minreal(G_m_inv * G_m_ninv0 - G_m0, verbose=False)
assert np.allclose(diff.num[0][0], 0.0)
print(G_m_inv)
print(G_m_ninv0)

### Feedforward and Reference Model

The feedforward controller and the resulting tracking dynamics are:

$$
G_{ff}(s) = \frac{F(s)}{\tilde{G}_m(s)}
= \frac{(10s + 1)(5s + 1)}{3(5s + 1)^2}
= \frac{10s + 1}{3(5s + 1)}
$$

$$
G_r(s) = F(s) \bar{G}_m(s) = \frac{-2s + 1}{(5s + 1)^2} e^{-4s}
$$

$G_r(s)$ is the reference model: the setpoint response obtained with the
feedforward alone when the model is perfect, $G_r(s) = G_{ff}(s) G_m(s)$.
The feedback controller acts on the error between this reference
trajectory and the plant output.

In [ ]:
# Feedforward (minreal cancels the common (5s + 1) factor)
G_ff = con.minreal(F / G_m_inv, verbose=False)
print(G_ff)

In [ ]:
# Reference model (rational part)
G_r0 = F * G_m_ninv0
print(G_r0)

In [ ]:
# Check that G_ff * G_m0 equals the reference model rational part
diff = con.minreal(G_ff * G_m0 - G_r0, verbose=False)
assert np.allclose(diff.num[0][0], 0.0)
print(con.minreal(G_ff * G_m0, verbose=False))

In [ ]:
# Discrete-time feedforward and reference model (ZOH)
G_ff_d = con.c2d(G_ff, Ts, 'zoh')
G_r_d = con.ss(con.c2d(G_r0, Ts, 'zoh')) * z_delay

### Simulation 3: Setpoint Feedforward

The reference model is used as the reference filter ($F_r = G_r$), with
the setpoint feedforward $G_{ff}$.

In [ ]:
T_ff = make_closed_loop_system(
    G_c_d, G_p_d, F_r=G_r_d, G_ff=G_ff_d, name='setpoint_ff'
)
sim_ff = simulate(T_ff)
plot_sim_results(t, r, sim_ff, title='Anticipation de la consigne')

### Simulation 4: Plant-Model Mismatch in the Time Delay

The feedforward $G_{ff}$ and reference model $G_r$ are designed with the
model delay $\theta = 4$ s. Here the actual plant delay $\theta_p$ is
shorter or longer than the model delay. The feedforward then no longer
produces the reference trajectory exactly, so the error $e(t)$ is non-zero
during the setpoint change and the feedback controller has to act.

In [ ]:
def make_discrete_plant(theta_p):
    """Discrete-time plant (ZOH) with the rational part G_p0 and time
    delay theta_p (must be a multiple of Ts)."""
    n = round(theta_p / Ts)
    assert np.isclose(n * Ts, theta_p), "delay must be a multiple of Ts"
    delay = con.ss(con.tf([1], [1] + [0] * n, Ts))
    return delay * con.ss(con.c2d(G_p0, Ts, 'zoh'))


# Actual plant delays (s)
plant_delays = [3.0, theta, 5.0]

sims_mm = {}
for theta_p in plant_delays:
    T_mm = make_closed_loop_system(
        G_c_d, make_discrete_plant(theta_p), F_r=G_r_d, G_ff=G_ff_d,
        name='setpoint_ff_mismatch'
    )
    sims_mm[rf'$\theta_p = {theta_p:g}$ s'] = simulate(T_mm)

In [ ]:
def plot_sim_comparison(t, r, sims, title=None):
    """Plot the output, error and control signals of several
    simulations, given as a dictionary {label: sim}."""
    fig, axes = plt.subplots(3, 1, sharex=True, figsize=(7, 1 + 3*1.5))

    ax = axes[0]
    ax.plot(t, r, 'k-', lw=1, label='$r(t)$')
    for label, sim in sims.items():
        ax.plot(t, sim['y'], label=label)
    ax.plot(t, sim['r_m'], 'k--', lw=1, label='$r_m(t)$')
    ax.set_ylabel('$r(t)$, $y(t)$')
    ax.set_title(title)
    ax.legend(loc='lower right', fontsize='small')

    for label, sim in sims.items():
        axes[1].plot(t, sim['e'], label=label)
        axes[2].plot(t, sim['u'], label=label)
    axes[1].set_ylabel('$e(t)$')
    axes[2].set_ylabel('$u(t)$')
    axes[2].set_xlabel('Temps (s)')

    for ax in axes:
        ax.grid()
    plt.tight_layout()
    plt.show()


plot_sim_comparison(t, r, sims_mm, title='Plant-model mismatch in the time delay')